# Pseudo-Labeling Experiments

Moved from `train_classifier.ipynb` Sections 5-7.
Uses pseudo-labeled data to retrain the classifier and compares against a keyword baseline.

## Setup

In [ ]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

In [ ]:
from torchvision import models, transforms

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
import sys, json, random
from pathlib import Path

In [ ]:
sys.path.insert(0, str(Path.cwd().parent))
from classifier.dataset import ProductDataset
from classifier.categories import CATEGORY_MAP

In [ ]:
from classifier.train import train_model

DATA_DIR = Path.cwd().parent / "data" / "raw"
MODEL_DIR = DATA_DIR.parent / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOAD_FROM_DISK = True

## 1. Pseudo-label Unlabeled Images

In [ ]:
import subprocess, sys
from pathlib import Path
result = subprocess.run(
    [sys.executable, "-m", "classifier.pseudo_label"],
    cwd=Path.cwd().parent,
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## 2. Retrain with Pseudo-Labeled Data

Loads pseudo-label model from disk when available.

In [ ]:
train_ds2 = ProductDataset(DATA_DIR, split='train')
val_ds2   = ProductDataset(DATA_DIR, split='val')
test_ds2  = ProductDataset(DATA_DIR, split='test')
test_loader2 = DataLoader(test_ds2, 32, shuffle=False)
total_train = len(train_ds2)
total_val = len(val_ds2)
total_test = len(test_ds2)
print('Train:', total_train, 'Val:', total_val, 'Test:', total_test)
print('Classes:', train_ds2.classes)

MODEL2_PATH = MODEL_DIR / "resnet50_pseudo_v1.pt"

if LOAD_FROM_DISK and MODEL2_PATH.exists() and (MODEL_DIR / "history_pseudo_v1.json").exists():
    model2 = models.resnet50(weights=None)
    model2.fc = nn.Linear(2048, len(train_ds2.classes))
    model2.load_state_dict(torch.load(MODEL2_PATH, map_location=DEVICE, weights_only=True))
    model2.to(DEVICE)
    with open(MODEL_DIR / "history_pseudo_v1.json") as f:
        history2 = json.load(f)
    best_acc2 = max(history2["val_acc"])
    print(f"Loaded pseudo-label model from disk (best_acc={best_acc2:.2f}%)")
else:
    model2, history2, best_acc2 = train_model(train_ds2, val_ds2, DEVICE, epochs=20, unfreeze_layers=14)
    torch.save(model2.state_dict(), MODEL2_PATH)
    with open(MODEL_DIR / "history_pseudo_v1.json", "w") as f:
        json.dump(history2, f)
    print(f'Best validation accuracy: {best_acc2}')

In [ ]:
if LOAD_FROM_DISK and (MODEL_DIR / "test_preds2.npy").exists():
    all_preds2 = np.load(MODEL_DIR / "test_preds2.npy").tolist()
    all_labels2 = np.load(MODEL_DIR / "test_labels2.npy").tolist()
    print(f"Loaded cached pseudo-label predictions from {MODEL_DIR}")
else:
    model2.eval()
    all_preds2 = []
    all_labels2 = []
    with torch.no_grad():
        for images, labels in tqdm(test_loader2, desc='Testing'):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model2(images)
            _, predicted = torch.max(outputs, 1)
            all_preds2.extend(predicted.cpu().numpy())
            all_labels2.extend(labels.cpu().numpy())
    np.save(MODEL_DIR / "test_preds2.npy", np.array(all_preds2))
    np.save(MODEL_DIR / "test_labels2.npy", np.array(all_labels2))

cm2 = confusion_matrix(all_labels2, all_preds2)
plt.figure(figsize=(8, 6))
sns.heatmap(cm2, annot=True, fmt='d', cmap='Blues', xticklabels=train_ds2.classes, yticklabels=train_ds2.classes)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - After Pseudo-labeling')
plt.show()
print(classification_report(all_labels2, all_preds2, target_names=train_ds2.classes))

## 3. Comparison vs Phase 5 Keyword Baseline

Run the keyword-based classifier from `extraction.py` on the same test set
to compare accuracy.

In [ ]:
import subprocess, sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, "-m", "classifier.compare_baselines"],
    cwd=Path.cwd().parent,
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# ML accuracy from previous sections -- fill in after running above
print()
print("Test accuracies to compare:")
print("  ML (before pseudo-labels):   <fill in>")
print("  ML (after pseudo-labels):    <fill in>")
print("  Keyword (product name):      <see above>")